# Bulk RNA-seq Explorer

Interactive companion to the HTML report.

## Setup

In [ ]:
suppressPackageStartupMessages({
  library(yaml); library(readr); library(dplyr); library(tidyr)
  library(tibble); library(ggplot2); library(scales)
})

`%||%` <- function(a, b) if (is.null(a)) b else a

PROJECT_ROOT <- Sys.getenv("PROJECT_ROOT", normalizePath("..", mustWork = TRUE))
CONFIG_PATH  <- file.path(PROJECT_ROOT, "config", "config.yaml")

rp <- function(...) file.path(PROJECT_ROOT, ...)

cfg       <- yaml::read_yaml(CONFIG_PATH)
samples   <- read_tsv(rp(cfg$input$samples_tsv   %||% "config/samples.tsv"),   show_col_types = FALSE)
contrasts <- read_tsv(rp(cfg$input$contrasts_tsv %||% "config/contrasts.tsv"), show_col_types = FALSE)

no_results <- function(msg = "No results available for this section.") {
  message(msg); invisible(NULL)
}

safe_read <- function(path, sep = ",") {
  if (is.null(path) || !file.exists(path)) return(NULL)
  df <- tryCatch(
    if (sep == "\t") read_tsv(path, show_col_types = FALSE)
    else             read_csv(path, show_col_types = FALSE),
    error = function(e) NULL)
  if (is.null(df) || nrow(df) == 0) return(df)
  df
}

contrast_row <- function(cid) {
  out <- contrasts[contrasts$contrast_id == cid, , drop = FALSE]
  if (nrow(out) == 0) return(NULL)
  as.list(out[1, ])
}

# Pick a contrast to explore
CONTRAST <- contrasts$contrast_id[1]
cat("Project :", basename(PROJECT_ROOT),
    "\nContrast:", CONTRAST,
    "\nAvailable:", paste(contrasts$contrast_id, collapse = ", "), "\n")

## 1. QC

MultiQC-aggregated metrics (`results/qc/qc_summary.tsv`). Sample quality (degradation, contamination, library-prep deviations).

In [ ]:
qc <- safe_read(rp("results", "qc", "qc_summary.tsv"), sep = "\t")
qc

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 3.5)
if (is.null(qc) || nrow(qc) == 0 || !"assigned_reads" %in% names(qc)) {
  no_results("Per-sample assigned-read totals not available.")
} else {
  df <- qc |>
    dplyr::mutate(sample    = factor(sample, levels = sample),
                  condition = as.character(condition))
  ggplot(df, aes(x = sample, y = assigned_reads, fill = condition)) +
    geom_col() +
    scale_y_continuous(labels = scales::comma) +
    labs(x = NULL, y = "Assigned reads (sum of counts)") +
    theme_minimal(base_size = 11) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
}

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5)
if (is.null(qc) || nrow(qc) == 0) {
  no_results("QC rate metrics not available.")
} else {
  rate_cols <- intersect(
    c("mapping_rate", "uniquely_mapped_pct", "rrna_pct",
      "duplication", "dupradar_slope", "exonic_pct"),
    names(qc)
  )
  rate_cols <- rate_cols[vapply(rate_cols, function(c) any(!is.na(qc[[c]])), logical(1))]
  if (length(rate_cols) == 0) {
    no_results("All rate metrics are NA.")
  } else {
    df <- qc |>
      dplyr::select(sample, condition, dplyr::all_of(rate_cols)) |>
      tidyr::pivot_longer(dplyr::all_of(rate_cols), names_to = "metric", values_to = "value") |>
      dplyr::mutate(sample = factor(sample, levels = unique(sample)))
    ggplot(df, aes(x = sample, y = value, fill = condition)) +
      geom_col() +
      facet_wrap(~ metric, scales = "free_y") +
      labs(x = NULL, y = NULL) +
      theme_minimal(base_size = 11) +
      theme(axis.text.x = element_text(angle = 45, hjust = 1))
  }
}

### Sample handling indicators

Cohort-relative outliers in these metrics indicate sample-specific preparation issues:

- `gene_body_5_3_bias` — RNA integrity indicator. ~1.0 = normal, <0.5 = 3' bias from transcript truncation.
- `intronic_pct`, `intergenic_pct` — excess pre-mRNA or genomic DNA carryover.
- `insert_size_median` — fragmentation indicator.
- `gc_content_pct` — foreign-organism contamination or adapter dimers.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5)
if (is.null(qc) || nrow(qc) == 0) {
  no_results("Handling-indicator metrics not available.")
} else {
  cols <- intersect(
    c("gene_body_5_3_bias", "intronic_pct", "intergenic_pct",
      "insert_size_median", "gc_content_pct"),
    names(qc)
  )
  cols <- cols[vapply(cols, function(c) any(!is.na(qc[[c]])), logical(1))]
  if (length(cols) == 0) {
    no_results("Handling indicators all NA — confirm MultiQC modules present.")
  } else {
    df <- qc |>
      dplyr::select(sample, condition, dplyr::all_of(cols)) |>
      tidyr::pivot_longer(dplyr::all_of(cols),
                          names_to = "metric", values_to = "value") |>
      dplyr::mutate(sample = factor(sample, levels = unique(sample)))
    print(
      ggplot(df, aes(x = sample, y = value, fill = condition)) +
        geom_col() +
        facet_wrap(~ metric, scales = "free_y") +
        labs(x = NULL, y = NULL) +
        theme_minimal(base_size = 11) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1))
    )
  }
}

if (!is.null(qc) && "strandedness" %in% names(qc) && any(!is.na(qc$strandedness))) {
  calls <- unique(stats::na.omit(qc$strandedness))
  if (length(calls) == 1) {
    cat(sprintf("Strandedness sanity check: all samples called as %s.\n", calls))
  } else {
    tab <- table(qc$strandedness)
    cat("Strandedness mismatch across samples:",
        paste(sprintf("%s (n=%d)", names(tab), as.integer(tab)), collapse = ", "),
        "\n")
  }
}


## 2. Exploratory

PCA on VST top-500 variable genes; sample distances on all genes (Euclidean). Precomputed objects loaded from `results/exploratory/`.

In [ ]:
pca_obj <- readRDS(rp("results", "exploratory", "pca.rds"))
cor_obj <- readRDS(rp("results", "exploratory", "sample_correlation.rds"))
str(pca_obj, max.level = 1); cat("---\n"); str(cor_obj, max.level = 1)

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 5.5)
scores <- pca_obj$scores
ve     <- pca_obj$var_explained
pc1_lbl <- sprintf("PC1 (%.1f%%)", 100 * (ve[["PC1"]] %||% NA))
pc2_lbl <- sprintf("PC2 (%.1f%%)", 100 * (ve[["PC2"]] %||% NA))

ggplot(scores, aes(x = PC1, y = PC2, colour = condition, shape = as.factor(batch), label = sample)) +
  geom_point(size = 3) +
  geom_text(aes(label = sample), nudge_y = 0.04, size = 3, show.legend = FALSE) +
  labs(x = pc1_lbl, y = pc2_lbl, colour = "condition", shape = "batch") +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 4)
ve <- pca_obj$var_explained
scree <- data.frame(PC = factor(names(ve), levels = names(ve)),
                    pct = as.numeric(ve) * 100)
scree <- head(scree, min(10, nrow(scree)))

ggplot(scree, aes(x = PC, y = pct)) +
  geom_col(fill = "#4c7fb0") +
  labs(x = NULL, y = "% variance explained") +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 6.5)
M   <- cor_obj$dist
ord <- if (!is.null(cor_obj$hclust)) cor_obj$hclust$order else seq_len(nrow(M))
M   <- M[ord, ord, drop = FALSE]
df  <- as.data.frame(as.table(M))
names(df) <- c("sample_x", "sample_y", "distance")
df$sample_x <- factor(df$sample_x, levels = rownames(M))
df$sample_y <- factor(df$sample_y, levels = rownames(M))

ggplot(df, aes(x = sample_x, y = sample_y, fill = distance)) +
  geom_tile() +
  geom_text(aes(label = sprintf("%.1f", distance)), size = 3, colour = "grey20") +
  scale_fill_gradient(low = "#4c7fb0", high = "#ffffff",
                      limits = c(0, max(df$distance, na.rm = TRUE))) +
  labs(x = NULL, y = NULL, fill = "distance") +
  theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.5)
hc <- cor_obj$hclust
op <- graphics::par(mar = c(6, 1, 1, 1))
plot(hc, main = "", xlab = "", sub = "", ylab = "", yaxt = "n", hang = -1)
graphics::par(op)

## 3. Differential expression — `CONTRAST`

DESeq2 results. Edit `padj_cutoff` / `lfc_cutoff` in the next cell to re-threshold without re-running DESeq2. apeglm-shrunken LFC, B-H padj.

In [ ]:
padj_cutoff <- cfg$de$primary$padj    %||% 0.05
lfc_cutoff  <- cfg$de$primary$abs_lfc %||% 1

de_tbl  <- safe_read(rp("results", "de", CONTRAST, "deseq2_results.csv"))
deg_sum <- safe_read(rp("results", "de", CONTRAST, "deg_summary.tsv"), sep = "\t")

dds_path <- rp("results", "exploratory", "dds_vst.rds")
vst_mat  <- if (file.exists(dds_path)) {
  obj <- readRDS(dds_path)
  tryCatch(SummarizedExperiment::assay(obj), error = function(e) as.matrix(obj@assays@data[[1]]))
} else NULL

deg_sum

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 6)
v_df <- de_tbl |>
  dplyr::mutate(neglog10p = -log10(pmax(pvalue, 1e-300)),
                sig = !is.na(padj) & padj < padj_cutoff & abs(log2FoldChange) >= lfc_cutoff)

sig_df <- v_df |> dplyr::filter(sig)
ns_df  <- v_df |> dplyr::filter(!sig)
if (nrow(ns_df) > 3000) { set.seed(42); ns_df <- ns_df[sample.int(nrow(ns_df), 3000), ] }

ggplot() +
  geom_point(data = ns_df,  aes(log2FoldChange, neglog10p),
             colour = "grey70", alpha = 0.45, size = 1.1) +
  geom_point(data = sig_df, aes(log2FoldChange, neglog10p),
             colour = "#c0392b", alpha = 0.85, size = 1.6) +
  geom_vline(xintercept = c(-lfc_cutoff, lfc_cutoff), linetype = "dashed", colour = "grey40") +
  geom_hline(yintercept = -log10(padj_cutoff),       linetype = "dashed", colour = "grey40") +
  labs(title = CONTRAST, x = "log2FoldChange (apeglm)", y = "-log10(pvalue)",
       subtitle = sprintf("sig: padj<%.2g & |LFC|>=%.2g", padj_cutoff, lfc_cutoff)) +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 5.5)
ma_df <- de_tbl |>
  dplyr::mutate(sig = !is.na(padj) & padj < padj_cutoff & abs(log2FoldChange) >= lfc_cutoff)

ggplot(ma_df, aes(log10(pmax(baseMean, 1)), log2FoldChange, colour = sig)) +
  geom_point(alpha = 0.5, size = 1.2) +
  scale_colour_manual(values = c(`TRUE` = "#c0392b", `FALSE` = "grey60")) +
  geom_hline(yintercept = 0, colour = "grey40") +
  labs(x = "log10(baseMean)", y = "log2FoldChange (apeglm)",
       colour = sprintf("padj<%.2g & |LFC|>=%.2g", padj_cutoff, lfc_cutoff)) +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 9)
top <- de_tbl |>
  dplyr::filter(!is.na(padj), !is.na(log2FoldChange)) |>
  dplyr::arrange(padj) |>
  head(30)

rows <- intersect(top$gene_id, rownames(vst_mat))
M <- vst_mat[rows, , drop = FALSE]
label_map <- setNames(top$gene_name, top$gene_id)
rn <- ifelse(is.na(label_map[rows]) | !nzchar(label_map[rows]), rows, label_map[rows])
rownames(M) <- make.unique(rn)
Z <- t(scale(t(M)))

row_hc <- hclust(dist(Z),       method = "average")
col_hc <- hclust(dist(t(Z)),    method = "average")
long <- as.data.frame(as.table(Z))
names(long) <- c("gene", "sample", "z")
long$gene   <- factor(long$gene,   levels = rownames(Z)[row_hc$order])
long$sample <- factor(long$sample, levels = colnames(Z)[col_hc$order])

ggplot(long, aes(sample, gene, fill = z)) +
  geom_tile() +
  scale_fill_gradient2(low = "#2c7bb6", mid = "white", high = "#d7191c", midpoint = 0) +
  labs(x = NULL, y = NULL, fill = "z-score") +
  theme_minimal(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        axis.text.y = element_text(size = 9))

In [ ]:
de_tbl |>
  dplyr::arrange(padj) |>
  dplyr::select(dplyr::any_of(c("gene_id", "gene_name", "baseMean",
                                 "log2FoldChange", "lfcSE", "stat",
                                 "pvalue", "padj"))) |>
  head(30)

## 4. GSEA

Per-collection Top 10 up / Top 10 down NES bar plots.
Ranking metric: Wald statistic

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 6)
gsea_all <- safe_read(rp("results", "enrichment", CONTRAST, "gsea_combined.csv"))

.coll_entries <- cfg$enrichment$gsea$collections %||% list()
.coll_labels <- setNames(
  vapply(.coll_entries, function(x) x$label %||% x$id, character(1)),
  vapply(.coll_entries, function(x) x$id, character(1))
)
.collection_label <- function(id) {
  out <- unname(.coll_labels[id])
  ifelse(is.na(out), id, out)
}

.trunc <- function(x, n = 50) ifelse(nchar(x) > n, paste0(substr(x, 1, n-1), "..."), x)

if (is.null(gsea_all) || nrow(gsea_all) == 0) {
  no_results("GSEA returned no results for this contrast.")
} else {
  for (coll in unique(as.character(gsea_all$collection))) {
    df <- gsea_all |> dplyr::filter(collection == coll)
    if (nrow(df) == 0) next
    grp <- .collection_label(coll)
    ranked <- df |>
      dplyr::mutate(direction = dplyr::case_when(NES > 0 ~ "up", NES < 0 ~ "down", TRUE ~ NA_character_)) |>
      dplyr::filter(direction %in% c("up", "down")) |>
      dplyr::group_by(direction) |>
      dplyr::arrange(padj, .by_group = TRUE) |>
      dplyr::slice_head(n = 10) |>
      dplyr::ungroup() |>
      dplyr::mutate(label = .trunc(pathway))
    ranked$label <- factor(ranked$label, levels = rev(unique(ranked$label)))
    print(
      ggplot(ranked, aes(NES, label, fill = direction)) +
        geom_col() +
        scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6")) +
        geom_vline(xintercept = 0, colour = "grey40") +
        labs(x = "NES", y = NULL,
             title = sprintf("%s — Top 10 up / Top 10 down (by padj)", grp)) +
        theme_minimal(base_size = 11) +
        theme(axis.text.y = element_text(size = 9))
    )
  }
}

### Enrichment plot

Run the **setup** cell, then change the pathway name in the **plot** cell and run.

In [ ]:
## --- GSEA enrichment-plot helpers (run once) -----------------------------
suppressPackageStartupMessages({ library(fgsea); library(msigdbr) })

.ranking <- cfg$enrichment$gsea$ranking %||% "stat"
.species <- cfg$enrichment$species      %||% "Homo sapiens"

.gsea_combined <- safe_read(rp("results", "enrichment", CONTRAST, "gsea_combined.csv"))

.build_ranks <- function(de_res, method) {
  if (method == "stat") {
    rdf <- de_res |> dplyr::filter(!is.na(gene_name), !is.na(stat)) |>
      dplyr::group_by(gene_name) |> dplyr::summarise(metric = mean(stat), .groups = "drop")
  } else if (method == "signed_p") {
    rdf <- de_res |> dplyr::filter(!is.na(gene_name), !is.na(log2FoldChange), !is.na(pvalue)) |>
      dplyr::mutate(metric = sign(log2FoldChange) * -log10(pmax(pvalue, 1e-300))) |>
      dplyr::filter(is.finite(metric)) |>
      dplyr::group_by(gene_name) |> dplyr::summarise(metric = mean(metric), .groups = "drop")
  } else stop("Unsupported ranking: ", method)
  rdf <- dplyr::arrange(rdf, dplyr::desc(metric))
  setNames(rdf$metric, rdf$gene_name)
}

.get_pathways <- function(coll_str, species) {
  parts    <- unlist(strsplit(coll_str, ":"))
  coll     <- parts[1]
  subcoll  <- if (length(parts) > 1) paste(parts[-1], collapse = ":") else NULL
  m <- if (!is.null(subcoll)) msigdbr(species = species, collection = coll, subcollection = subcoll)
       else                   msigdbr(species = species, collection = coll)
  split(m$gene_symbol, m$gs_name)
}

.pathway_cache <- new.env(parent = emptyenv())

.detect_collection <- function(pathway) {
  # Fast path: prefix match on well-known MSigDB naming conventions.
  prefixes <- c(HALLMARK_ = "H", REACTOME_ = "C2:CP:REACTOME",
                WP_ = "C2:CP:WIKIPATHWAYS", PID_ = "C2:CP:PID",
                BIOCARTA_ = "C2:CP:BIOCARTA")
  for (p in names(prefixes)) if (startsWith(pathway, p)) return(unname(prefixes[[p]]))
  # Slow path: look it up in the combined GSEA output.
  if (!is.null(.gsea_combined)) {
    hit <- .gsea_combined[.gsea_combined$pathway == pathway, "collection", drop = TRUE]
    if (length(hit) > 0) return(as.character(hit[1]))
  }
  stop(sprintf("Could not infer collection for '%s'. Pass collection = explicitly.", pathway))
}

.ranks <- .build_ranks(de_tbl, .ranking)

# Public helpers ----------------------------------------------------------

plot_gsea <- function(pathway, collection = NULL) {
  if (is.null(collection)) collection <- .detect_collection(pathway)
  if (is.null(.pathway_cache[[collection]]))
    .pathway_cache[[collection]] <- .get_pathways(collection, .species)
  sets <- .pathway_cache[[collection]]
  if (!pathway %in% names(sets))
    stop(sprintf("'%s' not found in collection %s.", pathway, collection))
  genes <- sets[[pathway]]
  row <- if (!is.null(.gsea_combined))
    .gsea_combined[.gsea_combined$collection == collection & .gsea_combined$pathway == pathway, ]
  else NULL
  nes_padj <- if (!is.null(row) && nrow(row) == 1)
    sprintf(" · NES=%.2f · padj=%.2g", row$NES, row$padj) else ""
  options(repr.plot.width = 8, repr.plot.height = 5)
  fgsea::plotEnrichment(genes, .ranks) +
    ggplot2::labs(title = pathway,
                  subtitle = sprintf("%s · %s · ranking=%s · |set ∩ ranked|=%d%s",
                                     CONTRAST, collection, .ranking,
                                     sum(genes %in% names(.ranks)), nes_padj))
}

list_pathways <- function(query = NULL, n = 20) {
  if (is.null(.gsea_combined) || nrow(.gsea_combined) == 0) return(.gsea_combined)
  all <- .gsea_combined |>
    dplyr::select(collection, pathway, NES, padj, size)
  if (!is.null(query))
    all <- all[grepl(query, all$pathway, ignore.case = TRUE), ]
  dplyr::arrange(all, padj) |> head(n)
}

show_leading <- function(pathway, collection = NULL) {
  if (is.null(collection)) collection <- .detect_collection(pathway)
  if (is.null(.gsea_combined)) { message("GSEA combined CSV not found."); return(invisible()) }
  row <- .gsea_combined[.gsea_combined$collection == collection & .gsea_combined$pathway == pathway, ]
  if (nrow(row) == 0) { message("Pathway not in saved GSEA results."); return(invisible()) }
  le <- strsplit(as.character(row$leadingEdge[1]), ";", fixed = TRUE)[[1]]
  cat(sprintf("%s · %s\nNES=%.3f · padj=%.3g · size=%d · leadingEdge (%d):\n\n",
              pathway, collection, row$NES, row$padj, row$size, length(le)))
  cat(paste(le, collapse = ", "), "\n")
}

invisible(NULL)  # keep setup cell quiet

In [ ]:
## --- Plot (edit the pathway name and run) ------------------------------
plot_gsea("HALLMARK_INTERFERON_GAMMA_RESPONSE")

# Collection auto-detected from name prefix. Override if ambiguous:
# plot_gsea("MY_CUSTOM_SET", collection = "C2:CGP")

In [ ]:
## --- Browse / search pathways actually available for this contrast ------
# Top 20 pathways across all collections (by padj):
list_pathways()

# Filter by keyword (case-insensitive):
# list_pathways("interferon")
# list_pathways("apop", n = 50)

# Leading-edge genes for a specific pathway:
# show_leading("HALLMARK_INTERFERON_GAMMA_RESPONSE")

## 5. ORA

Per-database Top 10 up / Top 10 down by p.adjust from `ora_combined.csv`.

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 7)
ora <- safe_read(rp("results", "enrichment", CONTRAST, "ora_combined.csv"))

.db_entries <- cfg$enrichment$ora$databases %||% list()
.db_labels <- setNames(
  vapply(.db_entries, function(x) x$label %||% x$id, character(1)),
  vapply(.db_entries, function(x) x$id, character(1))
)
.database_label <- function(id) {
  out <- unname(.db_labels[id])
  ifelse(is.na(out), id, out)
}

if (is.null(ora) || nrow(ora) == 0) {
  no_results("ORA returned no results.")
} else {
  for (db in unique(ora$database)) {
    show_df <- ora |>
      dplyr::filter(database == db, direction %in% c("up", "down")) |>
      dplyr::group_by(direction) |>
      dplyr::arrange(p.adjust, .by_group = TRUE) |>
      dplyr::slice_head(n = 10) |>
      dplyr::ungroup() |>
      dplyr::mutate(
        neglog10p   = -log10(pmax(p.adjust, 1e-300)),
        Description = ifelse(is.na(Description) | !nzchar(Description), ID, Description),
        Description = .trunc(Description, 50))
    if (nrow(show_df) == 0) next
    print(
      ggplot(show_df, aes(neglog10p, reorder(Description, neglog10p), fill = direction)) +
        geom_col(position = position_dodge(width = 0.8)) +
        scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6")) +
        labs(x = "-log10(p.adjust)", y = NULL,
             title = sprintf("%s — Top 10 up / Top 10 down", .database_label(db))) +
        theme_minimal(base_size = 11)
    )
  }
}

## 6. TFEA (CollecTRI / ULM)

Top TFs by |score| from `results/tfea/<contrast>/tf_scores.tsv`.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 8)
tf <- safe_read(rp("results", "tfea", CONTRAST, "tf_scores.tsv"), sep = "\t")
if (is.null(tf) || nrow(tf) == 0) {
  no_results("TFEA results not available.")
} else {
  top <- tf |> dplyr::slice_max(abs(score), n = 30) |>
    dplyr::mutate(tf = forcats::fct_reorder(source, score))
  ggplot(top, aes(score, tf, fill = score > 0)) +
    geom_col() +
    scale_fill_manual(values = c(`TRUE` = "#c0392b", `FALSE` = "#2c7bb6"), guide = "none") +
    labs(title = paste("Top TFEA —", CONTRAST), y = NULL) +
    theme_minimal(base_size = 11)
}

## 7. Pathway activity (PROGENy)

Contrast-level pathway activity inferred by decoupler MLM on the DESeq2 Wald statistic against the PROGENy network. Positive score → pathway up in numerator; negative → up in denominator.

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 6)
pw <- safe_read(rp("results", "progeny", CONTRAST, "progeny_scores.tsv"), sep = "\t")

if (is.null(pw) || nrow(pw) == 0 ||
    !all(c("pathway", "score") %in% names(pw)) ||
    all(is.na(pw$score))) {
  no_results("PROGENy scores unavailable or all-NA.")
} else {
  df <- pw |>
    dplyr::filter(!is.na(score)) |>
    dplyr::mutate(direction = ifelse(score > 0, "up", "down"),
                  pathway   = factor(pathway, levels = pathway[order(score)]))
  ggplot(df, aes(x = score, y = pathway, fill = direction)) +
    geom_col() +
    scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6"), guide = "none") +
    geom_vline(xintercept = 0, colour = "grey40") +
    labs(x = "MLM activity score", y = NULL,
         title = "PROGENy pathway activity",
         subtitle = sprintf("%s — input: DESeq2 Wald stat", CONTRAST)) +
    theme_minimal(base_size = 12)
}

### Score table (sorted by |score|)

In [ ]:
if (is.null(pw) || nrow(pw) == 0 || !"score" %in% names(pw)) {
  no_results("PROGENy table unavailable.")
} else {
  pw |> dplyr::arrange(dplyr::desc(abs(score)))
}

## 8. cMap (L2S2)

Top perturbagens by |score| from `results/cmap/<contrast>/l2s2_hits.tsv`.

### L2S2 input gene signatures

In [ ]:
top_up   <- cfg$cmap$top_up   %||% 250
top_down <- cfg$cmap$top_down %||% 250

if (is.null(de_tbl) || nrow(de_tbl) == 0) {
  no_results("DE results unavailable; L2S2 input signature cannot be reconstructed.")
} else {
  df <- de_tbl |>
    dplyr::filter(!is.na(gene_name), !is.na(log2FoldChange), !is.na(stat),
                  nzchar(as.character(gene_name)))
  up_tbl <- df |> dplyr::filter(log2FoldChange > 0) |>
    dplyr::slice_max(order_by = stat, n = top_up, with_ties = FALSE) |>
    dplyr::distinct(gene_name, .keep_all = TRUE) |>
    dplyr::mutate(direction = "up", input_rank = dplyr::row_number())
  down_tbl <- df |> dplyr::filter(log2FoldChange < 0) |>
    dplyr::slice_min(order_by = stat, n = top_down, with_ties = FALSE) |>
    dplyr::distinct(gene_name, .keep_all = TRUE) |>
    dplyr::mutate(direction = "down", input_rank = dplyr::row_number())
  sig_tbl <- dplyr::bind_rows(up_tbl, down_tbl) |>
    dplyr::select(dplyr::any_of(c("direction", "input_rank", "gene_id", "gene_name",
                                   "log2FoldChange", "stat", "pvalue", "padj")))
  cat(sprintf("L2S2 input: %d up / %d down genes (ranked by |Wald stat|)\n",
              sum(sig_tbl$direction == "up"), sum(sig_tbl$direction == "down")))
  sig_tbl
}


In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 8)
hits <- safe_read(rp("results", "cmap", CONTRAST, "l2s2_hits.tsv"), sep = "\t")
if (is.null(hits) || nrow(hits) == 0) {
  no_results("L2S2 query returned no hits.")
} else {
  top <- hits |>
    dplyr::arrange(dplyr::desc(abs(score %||% 0)), fdr) |>
    head(30) |>
    dplyr::mutate(label = ifelse(is.na(perturbagen) | !nzchar(perturbagen),
                                 paste0("row_", dplyr::row_number()), perturbagen),
                  label = factor(label, levels = rev(label)))
  ggplot(top, aes(score, label, fill = ifelse(score > 1, "induce", "reverse"))) +
    geom_col() +
    scale_fill_manual(values = c(induce = "#e67e22", reverse = "#2c7bb6"), name = "direction") +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey40") +
    labs(x = "odds ratio (score)", y = NULL, title = "Top 30 perturbagens") +
    theme_minimal(base_size = 11)
}

### Re-query L2S2 with custom top-N

Re-runs the L2S2 paired-enrichment query directly against the public API with a user-chosen signature size. Run the **setup** cell once, then tune `top_up` / `top_down` in the next cell.

In [ ]:
## --- L2S2 re-query helpers (run once) ----------------------------------
suppressPackageStartupMessages({ library(httr); library(jsonlite) })

L2S2_ENDPOINT <- Sys.getenv("L2S2_API_URL", "https://l2s2.maayanlab.cloud/graphql")

.l2s2_paired_query <- '
query PairEnrichmentQuery($genesUp: [String]!, $genesDown: [String]!) {
  currentBackground {
    pairedEnrich(genesUp: $genesUp, genesDown: $genesDown) {
      consensus { drug oddsRatio pvalue adjPvalue }
    }
  }
}'

.l2s2_post <- function(payload) {
  resp <- httr::POST(L2S2_ENDPOINT, httr::content_type_json(),
                     body = jsonlite::toJSON(payload, auto_unbox = TRUE))
  httr::stop_for_status(resp)
  j <- httr::content(resp, as = "parsed", simplifyVector = FALSE)
  if (!is.null(j$errors))
    stop("GraphQL errors: ", jsonlite::toJSON(j$errors, auto_unbox = TRUE))
  j
}

# L2S2's pairedEnrich returns drug names only; MoA lives on FdaCount and the
# web UI joins them client-side. Mirror that with one aliased fdaCount batch.
.l2s2_fetch_moas <- function(drugs) {
  if (!length(drugs)) return(setNames(character(), character()))
  parts <- vapply(seq_along(drugs), function(i)
    sprintf('a%d: fdaCount(perturbation: %s) { moa }',
            i - 1L, jsonlite::toJSON(drugs[i], auto_unbox = TRUE)),
    character(1))
  query <- paste0("query MoaLookup { ", paste(parts, collapse = " "), " }")
  j <- tryCatch(.l2s2_post(list(query = query)),
                error = function(e) { message("MoA lookup failed: ", conditionMessage(e)); NULL })
  if (is.null(j)) return(setNames(rep("", length(drugs)), drugs))
  vals <- vapply(seq_along(drugs), function(i) {
    moa <- j$data[[paste0("a", i - 1L)]]$moa
    if (is.null(moa)) "" else as.character(moa)
  }, character(1))
  setNames(vals, drugs)
}

query_l2s2 <- function(top_up = 250, top_down = 250) {
  if (is.null(de_tbl)) stop("de_tbl not loaded — run the DE section first.")
  df <- de_tbl |>
    dplyr::filter(!is.na(gene_name), nzchar(as.character(gene_name)),
                  !is.na(log2FoldChange), !is.na(stat))
  up <- df |> dplyr::filter(log2FoldChange > 0) |>
    dplyr::slice_max(order_by = stat, n = top_up, with_ties = FALSE) |>
    dplyr::pull(gene_name) |> as.character() |> unique()
  down <- df |> dplyr::filter(log2FoldChange < 0) |>
    dplyr::slice_min(order_by = stat, n = top_down, with_ties = FALSE) |>
    dplyr::pull(gene_name) |> as.character() |> unique()
  if (!length(up) || !length(down))
    stop(sprintf("signature incomplete (up=%d down=%d); paired enrichment needs both.",
                 length(up), length(down)))
  cat(sprintf("Signature: %d up / %d down — querying %s\n",
              length(up), length(down), L2S2_ENDPOINT))

  payload <- list(query = .l2s2_paired_query,
                  variables = list(genesUp = I(up), genesDown = I(down)))
  j <- .l2s2_post(payload)
  cons <- j$data$currentBackground$pairedEnrich$consensus
  if (length(cons) == 0) { cat("No consensus hits.\n"); return(data.frame()) }

  out <- do.call(rbind, lapply(cons, function(r) data.frame(
    perturbagen = r$drug      %||% NA_character_,
    score       = r$oddsRatio %||% NA_real_,
    pvalue      = r$pvalue    %||% NA_real_,
    fdr         = r$adjPvalue %||% NA_real_,
    stringsAsFactors = FALSE)))
  out <- out[order(out$fdr, -out$score), , drop = FALSE]
  rownames(out) <- NULL

  drugs <- unique(out$perturbagen[!is.na(out$perturbagen) & nzchar(out$perturbagen)])
  moa_map <- .l2s2_fetch_moas(drugs)
  out$moa <- ifelse(out$perturbagen %in% names(moa_map),
                    moa_map[out$perturbagen], "")
  out$rank <- seq_len(nrow(out))
  out[, c("rank", "perturbagen", "moa", "score", "pvalue", "fdr")]
}

plot_l2s2 <- function(hits, n = 30, title = NULL) {
  if (is.null(hits) || nrow(hits) == 0) { no_results("No L2S2 hits."); return(invisible()) }
  top <- head(hits, n) |>
    dplyr::mutate(label = ifelse(is.na(perturbagen) | !nzchar(perturbagen),
                                 paste0("row_", dplyr::row_number()), perturbagen),
                  label = factor(label, levels = rev(label)))
  options(repr.plot.width = 8.5, repr.plot.height = max(4, n * 0.25))
  ggplot(top, aes(score, label, fill = ifelse(score > 1, "induce", "reverse"))) +
    geom_col() +
    scale_fill_manual(values = c(induce = "#e67e22", reverse = "#2c7bb6"), name = "direction") +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey40") +
    labs(x = "odds ratio (score)", y = NULL,
         title = title %||% sprintf("Top %d perturbagens", nrow(top))) +
    theme_minimal(base_size = 11)
}

invisible(NULL)


In [ ]:
## --- Try different signature sizes (edit top_up / top_down) ------------
hits_custom <- query_l2s2(top_up = 100, top_down = 100)
plot_l2s2(hits_custom, n = 30, title = "Top 30 perturbagens (up=100, down=100)")

# head(hits_custom, 30)
# hits_500 <- query_l2s2(top_up = 500, top_down = 500); plot_l2s2(hits_500, n = 30)